<a href="https://colab.research.google.com/github/LeJ-04/web-datamining-semantics/blob/partieC-jeffrey/PartC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. Reasoning & Knowledge Graph Embeddings (KGE)

## C.1 Preparation of KGE Datasets (Train / Valid / Test)

Loading, Filtering and Splitting the Knowledge Graph

Knowledge Graph Embedding (KGE) models (such as TransE or ComplEx) learn vector representations of entities and relations based purely on the graph's topology.

To prepare our data for these models, we need to perform four critical operations.

1. **Filtering:**  
 KGE models strictly require **relational triples** `(head, relation, tail)` where all elements are URIs (Unique Resource Identifiers). We must filter out **Literals** (e.g., strings, numbers) and **Blank Nodes**.

In [1]:
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 10.2 MB/s eta 0:00:00


In [2]:
import os
import random
from rdflib import Graph, URIRef

In [3]:
graph_file = "clean_football_kb.ttl"
print(f"Chargement du graphe depuis {graph_file}...")
g = Graph()
g.parse(graph_file, format="turtle")
print(f"Graphe chargé ! Nombre total de triplets dans le graphe : {len(g)}")

Chargement du graphe depuis clean_football_kb.ttl...
Graphe chargé ! Nombre total de triplets dans le graphe : 64910


**Filtrage (isinstance(..., URIRef)) :**   
Les modèles mathématiques de KGE (Knowledge Graph Embeddings) comme TransE apprennent la structure du graphe. Ils ne savent pas lire du texte brut ou des nombres. On s'assure donc de ne garder que des triplets contenant des Identifiants (URI), c'est-à-dire des relations purement structurelles (ex: Joueur -> jouePour -> Equipe), en excluant les attributs comme les noms ou les âges (qui sont des Literal).

In [4]:
# 2. Extraction et filtrage des triplets
# Les modèles KGE classiques n'apprennent généralement que sur des relations entre Entités (URIs).
# On filtre donc pour exclure les valeurs littérales (noms en texte, nombres, etc.) et les BNodes.
triplets = []
for s, p, o in g:
    if isinstance(s, URIRef) and isinstance(p, URIRef) and isinstance(o, URIRef):
        # We store them as tuples for easier manipulation during OOV checking
        triplets.append((str(s), str(p), str(o)))

print(f"Total relational triples retained for KGE: {len(triplets)}")

Total relational triples retained for KGE: 64910


2. **Shuffling & Splitting:** We randomly shuffle the triples and split them into three sets:
   - **Training set (~80%):** Used by the model to learn the entity and relation embeddings.
   - **Validation set (~10%):** Used to tune hyperparameters and prevent overfitting.
   - **Testing set (~10%):** Held out until the end to evaluate the model's true performance.

In [5]:
# 3. Mélange aléatoire (Shuffle) pour garantir une distribution homogène
random.seed(42) # Fixer la seed pour la reproductibilité de l'expérience
random.shuffle(triplets)

**Le Split (80/10/10) :**   
C'est le standard en Machine Learning. Le Train sert à l'apprentissage, le Valid permet de vérifier en cours de route que le modèle ne fait pas de surapprentissage (overfitting), et le Test est utilisé à la toute fin pour évaluer sa performance réelle sur des données qu'il n'a jamais vues.

In [6]:
# 4. Calcul des indices pour la répartition 80% / 10% / 10%
total_triplets = len(triplets)
train_idx = int(total_triplets * 0.8)
valid_idx = int(total_triplets * 0.9)

train_triplets = triplets[:train_idx]
valid_triplets = triplets[train_idx:valid_idx]
test_triplets = triplets[valid_idx:]

print(f"Répartition des données :")
print(f" - Train : {len(train_triplets)} triplets")
print(f" - Valid : {len(valid_triplets)} triplets")
print(f" - Test  : {len(test_triplets)} triplets")

Répartition des données :
 - Train : 51928 triplets
 - Valid : 6491 triplets
 - Test  : 6491 triplets


3. **Preventing Out-Of-Vocabulary (OOV) Entities:**  
 A KGE model cannot evaluate an entity or relation it has never seen during training. After the initial split, we systematically check the Validation and Test sets. Any triple containing an entity or relation missing from the Training set is moved into the Training set. This guarantees 100% graph connectivity during evaluation.

In [7]:
# --- STEP 3: OUT-OF-VOCABULARY (OOV) PREVENTION ---
def get_vocab(triplet_list):
    """Returns a set of all entities and all relations in a given list of triplets."""
    entities, relations = set(), set()
    for s, p, o in triplet_list:
        entities.add(s)
        entities.add(o)
        relations.add(p)
    return entities, relations

In [8]:
train_entities, train_relations = get_vocab(train_triplets)

In [9]:
def fix_oov(target_triplets, train_triplets, train_entities, train_relations):
    """Moves triplets with unseen entities/relations to the train set."""
    fixed_target = []
    moved_count = 0
    for s, p, o in target_triplets:
        if s not in train_entities or o not in train_entities or p not in train_relations:
            # Missing in train: move this triplet to train
            train_triplets.append((s, p, o))
            train_entities.add(s)
            train_entities.add(o)
            train_relations.add(p)
            moved_count += 1
        else:
            fixed_target.append((s, p, o))
    return fixed_target, moved_count

In [10]:
# Fix Validation Set
valid_triplets, moved_val = fix_oov(valid_triplets, train_triplets, train_entities, train_relations)
# Fix Test Set
test_triplets, moved_test = fix_oov(test_triplets, train_triplets, train_entities, train_relations)

In [11]:
print(f"\nOOV Prevention Fixes:")
print(f" - Moved {moved_val} triples from Valid to Train.")
print(f" - Moved {moved_test} triples from Test to Train.")

print(f"\nFinal Data split breakdown:")
print(f" - Train : {len(train_triplets)} triples")
print(f" - Valid : {len(valid_triplets)} triples")
print(f" - Test  : {len(test_triplets)} triples")


OOV Prevention Fixes:
 - Moved 3402 triples from Valid to Train.
 - Moved 3248 triples from Test to Train.

Final Data split breakdown:
 - Train : 58578 triples
 - Valid : 3089 triples
 - Test  : 3243 triples


4. **Exporting:**  
 We save the splits into `.txt` files in a Tab-Separated Values (`\t`) format, which is the standard input format expected by KGE libraries like PyKEEN.

In [12]:
# --- STEP 4: FORMAT AND EXPORT THE DATASETS ---
def format_tsv(triplet_list):
    return [f"{s}\t{p}\t{o}\n" for s, p, o in triplet_list]

**L'export en .txt :**  
 Les bibliothèques de Machine Learning pour les graphes (comme PyKEEN) détestent parser de lourds fichiers XML ou Turtle. Elles demandent des fichiers texte basiques séparés par des tabulations (\t).

In [13]:
# 5. Création du dossier et sauvegarde des fichiers
dataset_dir = "kge_dataset"
os.makedirs(dataset_dir, exist_ok=True)

with open(os.path.join(dataset_dir, "train.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(train_triplets))

with open(os.path.join(dataset_dir, "valid.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(valid_triplets))

with open(os.path.join(dataset_dir, "test.txt"), "w", encoding="utf-8") as f:
    f.writelines(format_tsv(test_triplets))

print(f"\n✅ Successfully generated train.txt, valid.txt, and test.txt in the '{dataset_dir}/' directory.")


✅ Successfully generated train.txt, valid.txt, and test.txt in the 'kge_dataset/' directory.


## C.2 - Symbolic Reasoning with SWRL and Owlready2
=================================================  


Knowledge Graphs allow for explicitly defining rules to infer new, hidden
knowledge not directly stated in the dataset. This is called Symbolic Reasoning
and represents the "Neuro-Symbolic" aspect of modern AI.

We use the `owlready2` library to declare SWRL rules and rdflib SPARQL CONSTRUCT
to execute them (Pellet, the native SWRL reasoner, requires Java 25+ which may
not always be available).

Business Rules:
  1. If a Person plays for a Team, and that Team has a headCoach C,
     then the Person is coachedBy C.
     SWRL: Person(?p) ∧ Team(?t) ∧ Person(?c) ∧ playsFor(?p,?t) ∧ headCoach(?t,?c)
           → coachedBy(?p,?c)

  2. If a Person plays for a Team, and that Team is locatedIn a Country,
     then the Person competesIn that Country.
     SWRL: Person(?p) ∧ Team(?t) ∧ Country(?c) ∧ playsFor(?p,?t) ∧ locatedIn(?t,?c)
           → competesIn(?p,?c)

These inferred properties enrich the Knowledge Graph and can later be used
by the LLM in the RAG pipeline (e.g. "Which players compete in England?").

In [ ]:
!pip install owlready2

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import rdflib
from rdflib import RDF, OWL, Namespace
from rdflib.namespace import RDFS
from owlready2 import World, Thing, ObjectProperty, Imp
import os

In [ ]:
# =====================================================================
# --- 1. PRÉPARATION DU GRAPHE (RDFLIB) ---
# =====================================================================
print("=" * 60)
print("C.2 — Symbolic Reasoning with SWRL and Owlready2")
print("=" * 60)
print("\n1. Chargement et préparation du graphe Turtle...")

g = rdflib.Graph()
g.parse("clean_football_kb.ttl", format="turtle")

# IRIs exactes du Knowledge Graph football
ONTO_IRI = "http://www.example.org/football"
EX       = Namespace("http://www.example.org/football/")       # entités
EX_PROP  = Namespace("http://www.example.org/football/prop/")  # propriétés

# Déclaration OWL de l'ontologie
g.add((URIRef(ONTO_IRI), RDF.type, OWL.Ontology))

# Déclaration explicite des classes OWL
for cls in ["Person", "Team", "Country", "FootballPosition"]:
    g.add((EX[cls], RDF.type, OWL.Class))

# Déclaration des propriétés (existantes + inférées)
for p in ["playsFor", "headCoach", "locatedIn",
          "nationality", "playsPosition", "coachedBy", "competesIn"]:
    g.add((EX_PROP[p], RDF.type, OWL.ObjectProperty))

# Domaine / portée des propriétés inférées
g.add((EX_PROP["coachedBy"],  RDFS.domain, EX["Person"]))
g.add((EX_PROP["coachedBy"],  RDFS.range,  EX["Person"]))
g.add((EX_PROP["competesIn"], RDFS.domain, EX["Person"]))
g.add((EX_PROP["competesIn"], RDFS.range,  EX["Country"]))

# Typage de toutes les instances comme OWL.NamedIndividual (requis par owlready2)
IGNORE = {OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty,
          OWL.AnnotationProperty, OWL.Ontology, OWL.NamedIndividual}
for s, p, o in list(g.triples((None, RDF.type, None))):
    if o not in IGNORE and isinstance(s, URIRef):
        g.add((s, RDF.type, OWL.NamedIndividual))

count_ni = len(list(g.triples((None, RDF.type, OWL.NamedIndividual))))
print(f"   -> {count_ni} entités taguées NamedIndividual")

# Contrôle des propriétés clés
for prop in ["playsFor", "locatedIn", "headCoach"]:
    triples = list(g.triples((None, EX_PROP[prop], None)))
    ex = triples[0] if triples else None
    print(f"   -> '{prop}' : {len(triples)} triplets" +
          (f"  (ex: {ex[0].split('/')[-1]} → {ex[2].split('/')[-1]})" if ex else " ⚠️ AUCUN"))

# Sérialisation en XML/OWL (format attendu par owlready2)
xml_file = os.path.abspath("clean_football_kb_typed.xml")
g.serialize(xml_file, format="xml")
print(f"   -> XML intermédiaire : {xml_file}")

In [ ]:
# =====================================================================
# --- 2. INITIALISATION DU SCHÉMA OWLREADY2 ---
# =====================================================================
print("\n2. Initialisation du schéma Owlready2...")

my_world = World()
onto     = my_world.get_ontology(f"file://{xml_file}").load()

# Résolution des classes (les noms courts sont utilisables dans les règles SWRL)
Person  = onto.search_one(iri=str(EX["Person"]))
Team    = onto.search_one(iri=str(EX["Team"]))
Country = onto.search_one(iri=str(EX["Country"]))

if not all([Person, Team, Country]):
    print("   ⚠️  Classes non trouvées. Diagnostic des 5 premières IRIs :")
    for i, e in enumerate(list(onto.individuals())[:5]):
        print(f"     {e.iri}")
    raise RuntimeError("Vérifiez que les namespaces EX et EX_PROP sont corrects.")

print(f"   -> Person={Person.name} | Team={Team.name} | Country={Country.name}")

# Namespace des propriétés (différent du base_iri de l'ontologie)
prop_ns = my_world.get_namespace("http://www.example.org/football/prop/")

In [ ]:
# =====================================================================
# --- 3. DÉCLARATION DES RÈGLES SWRL ---
# =====================================================================
print("\n3. Déclaration des règles SWRL (owlready2.Imp)...")

with onto:
    # Règle 1 — Inférer le coach d'un joueur
    rule1 = Imp()
    rule1.set_as_rule(
        "Person(?p), Team(?t), Person(?c), "
        "playsFor(?p, ?t), headCoach(?t, ?c) -> coachedBy(?p, ?c)",
        namespaces=[onto, prop_ns]
    )
    # Règle 2 — Inférer le pays de compétition d'un joueur
    rule2 = Imp()
    rule2.set_as_rule(
        "Person(?p), Team(?t), Country(?c), "
        "playsFor(?p, ?t), locatedIn(?t, ?c) -> competesIn(?p, ?c)",
        namespaces=[onto, prop_ns]
    )

print("   -> Règle 1 : Person(?p) ∧ Team(?t) ∧ Person(?c)  ∧ playsFor(?p,?t) ∧ headCoach(?t,?c) → coachedBy(?p,?c)")
print("   -> Règle 2 : Person(?p) ∧ Team(?t) ∧ Country(?c) ∧ playsFor(?p,?t) ∧ locatedIn(?t,?c) → competesIn(?p,?c)")
print("   ✓ 2 règles SWRL enregistrées dans l'ontologie.")

In [ ]:
# =====================================================================
# --- 4. APPLICATION DES RÈGLES (SPARQL CONSTRUCT) ---
# =====================================================================
# owlready2 délègue l'exécution des règles SWRL au raisonneur Pellet (Java).
# Pellet requiert Java 25+ (class file 69.0). Si cette version n'est pas
# disponible, on applique les mêmes règles via SPARQL CONSTRUCT sur rdflib,
# ce qui est sémantiquement équivalent — c'est d'ailleurs ce que Pellet
# effectue en interne sur les données RDF.
print("\n4. Application des règles via SPARQL CONSTRUCT...")

g.bind("ex",   EX)
g.bind("prop", EX_PROP)
g.bind("owl",  OWL)

SPARQL_RULE1 = """
PREFIX ex:   <http://www.example.org/football/>
PREFIX prop: <http://www.example.org/football/prop/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
CONSTRUCT { ?p prop:coachedBy ?c . }
WHERE {
    ?p  rdf:type       ex:Person .
    ?t  rdf:type       ex:Team   .
    ?c  rdf:type       ex:Person .
    ?p  prop:playsFor  ?t .
    ?t  prop:headCoach ?c .
}
"""

SPARQL_RULE2 = """
PREFIX ex:   <http://www.example.org/football/>
PREFIX prop: <http://www.example.org/football/prop/>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
CONSTRUCT { ?p prop:competesIn ?c . }
WHERE {
    ?p  rdf:type        ex:Person  .
    ?t  rdf:type        ex:Team    .
    ?c  rdf:type        ex:Country .
    ?p  prop:playsFor   ?t .
    ?t  prop:locatedIn  ?c .
}
"""

# FIX : séparer query / add / count pour un décompte correct
new_triples_1 = list(g.query(SPARQL_RULE1))
for triple in new_triples_1:
    g.add(triple)
count_coached = len(new_triples_1)

new_triples_2 = list(g.query(SPARQL_RULE2))
for triple in new_triples_2:
    g.add(triple)
count_competes = len(new_triples_2)

print(f"   ✓ Règle 1 (coachedBy)  → {count_coached}  nouveaux triplets inférés")
print(f"   ✓ Règle 2 (competesIn) → {count_competes} nouveaux triplets inférés")

In [ ]:
# =====================================================================
# --- 5. VÉRIFICATION DES CONNAISSANCES INFÉRÉES ---
# =====================================================================
print("\n5. Vérification des nouvelles connaissances inférées...")

print("\n   ── Exemples : competesIn ──")
q_ex_competes = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?player ?country WHERE { ?player prop:competesIn ?country . } LIMIT 5
"""
for row in g.query(q_ex_competes):
    player  = str(row.player).split('/')[-1]
    country = str(row.country).split('/')[-1]
    print(f"     {player} compète en {country}")

print("\n   ── Exemples : coachedBy ──")
q_ex_coached = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?player ?coach WHERE { ?player prop:coachedBy ?coach . } LIMIT 5
"""
for row in g.query(q_ex_coached):
    player = str(row.player).split('/')[-1]
    coach  = str(row.coach).split('/')[-1]
    print(f"     {player} est coaché par {coach}")

print("\n   ── Joueurs par pays de compétition (top 5) ──")
q_by_country = """
PREFIX prop: <http://www.example.org/football/prop/>
SELECT ?country (COUNT(?player) AS ?nb)
WHERE { ?player prop:competesIn ?country . }
GROUP BY ?country ORDER BY DESC(?nb) LIMIT 5
"""
for row in g.query(q_by_country):
    country = str(row.country).split('/')[-1]
    print(f"     {country} : {int(row.nb)} joueurs")

In [41]:
# =====================================================================
# --- 6. SAUVEGARDE DU GRAPHE AUGMENTÉ ---
# =====================================================================
print("\n6. Sauvegarde du graphe enrichi...")

augmented_ttl = "augmented_football_kb.ttl"
augmented_xml = "augmented_football_kb.xml"

g.serialize(augmented_ttl, format="turtle")   # format humain
onto.save(file=augmented_xml, format="rdfxml")  # format owlready2

total_triples = len(list(g.triples((None, None, None))))

print(f"""
{'='*60}
✅  C.2 TERMINÉ — Résumé du raisonnement symbolique
{'='*60}
  Graphe initial         : {total_triples - count_coached - count_competes} triplets
  Règles SWRL déclarées  : 2 (owlready2.Imp)
  Nouveaux coachedBy     : {count_coached}
  Nouveaux competesIn    : {count_competes}
  ─────────────────────────────────────────────
  Total triplets final   : {total_triples}
  Fichier Turtle         : {augmented_ttl}  ({os.path.getsize(augmented_ttl):,} octets)
  Fichier XML/OWL        : {augmented_xml}  ({os.path.getsize(augmented_xml):,} octets)
""")

C.2 — Symbolic Reasoning with SWRL and Owlready2

1. Chargement et préparation du graphe Turtle...
   -> 776 entités taguées NamedIndividual
   -> 'playsFor' : 647 triplets  (ex: Aaron_Bott → Nottingham_Forest_FC)
   -> 'locatedIn' : 20 triplets  (ex: Fulham_FC → England)
   -> 'headCoach' : 20 triplets  (ex: Fulham_FC → Marco_Silva)
   -> XML intermédiaire : /content/clean_football_kb_typed.xml

2. Initialisation du schéma Owlready2...
   -> Person=Person | Team=Team | Country=Country

3. Déclaration des règles SWRL (owlready2.Imp)...
   -> Règle 1 : Person(?p) ∧ Team(?t) ∧ Person(?c)  ∧ playsFor(?p,?t) ∧ headCoach(?t,?c) → coachedBy(?p,?c)
   -> Règle 2 : Person(?p) ∧ Team(?t) ∧ Country(?c) ∧ playsFor(?p,?t) ∧ locatedIn(?t,?c) → competesIn(?p,?c)
   ✓ 2 règles SWRL enregistrées dans l'ontologie.

4. Application des règles via SPARQL CONSTRUCT...
   ✓ Règle 1 (coachedBy)  → 647  nouveaux triplets inférés
   ✓ Règle 2 (competesIn) → 647 nouveaux triplets inférés

5. Vérification des no

## C.3 — Knowledge Graph Embeddings : Training TransE & ComplEx
# =====================================================================
PyKEEN est la librairie de référence pour les KGE en Python.
Elle gère nativement : chargement TSV, entraînement, évaluation, sauvegarde des embeddings et calcul des métriques.

Deux modèles entraînés :
- TransE   — modèle translationnel (h + r ≈ t dans l'espace vectoriel)
- ComplEx  — modèle bilinéaire dans l'espace complexe (meilleur sur les relations asymétriques comme playsFor, coachedBy)

In [42]:
# Installation (à exécuter une seule fois) ─────────────────────────
# !pip install pykeen torch --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00


In [ ]:
import torch
import pykeen
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from pykeen.models import TransE, ComplEx
import pandas as pd
import matplotlib.pyplot as plt
import os, json, time

print(f"PyKEEN version : {pykeen.__version__}")
print(f"PyTorch version : {torch.__version__}")
print(f"GPU disponible  : {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé  : {device}")

In [ ]:
# =====================================================================
# --- 1. CHARGEMENT DES SPLITS ---
# =====================================================================
print("\n1. Chargement des splits KGE...")

# Chemins vers les fichiers générés en C.1
TRAIN_PATH = "train.txt"
VALID_PATH  = "valid.txt"
TEST_PATH   = "test.txt"

# TriplesFactory lit directement le format TSV (head \t relation \t tail)
# entity_to_id et relation_to_id sont partagés entre les 3 splits
# pour garantir la cohérence du vocabulaire (pas d'entités OOV)
tf_train = TriplesFactory.from_path(
    TRAIN_PATH,
    delimiter="\t"
)
tf_valid = TriplesFactory.from_path(
    VALID_PATH,
    delimiter="\t",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id,
)
tf_test = TriplesFactory.from_path(
    TEST_PATH,
    delimiter="\t",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id,
)

print(f"   Train  : {tf_train.num_triples:>6,} triplets")
print(f"   Valid  : {tf_valid.num_triples:>6,} triplets")
print(f"   Test   : {tf_test.num_triples:>6,} triplets")
print(f"   Entités    : {tf_train.num_entities:,}")
print(f"   Relations  : {tf_train.num_relations:,}")

In [ ]:
# =====================================================================
# --- 2. CONFIGURATION DES MODÈLES ---
# =====================================================================
# Hyperparamètres choisis pour un bon équilibre qualité / temps
# sur un graphe de taille moyenne (~600 entités, ~10 relations).
#
#   embedding_dim  : taille du vecteur de représentation (128 est standard)
#   epochs         : 200 est suffisant pour converger sur ce dataset
#   batch_size     : 256 pour stabiliser le gradient
#   lr             : 0.001 (Adam optimizer, valeur par défaut recommandée)
# =====================================================================

EMBEDDING_DIM = 128
EPOCHS        = 200
BATCH_SIZE    = 256
LR            = 0.001

MODEL_CONFIGS = {
    "TransE": {
        "model": "TransE",
        # TransE est le modèle fondateur (Bordes et al., 2013).
        # Il modélise chaque relation r comme une translation dans
        # l'espace vectoriel : h + r ≈ t.
        # Simple, efficace, mais peine sur les relations symétriques.
        "model_kwargs": {
            "embedding_dim": EMBEDDING_DIM,
        },
        "loss":    "MarginRankingLoss",
        "color":   "#E74C3C",
    },
    "ComplEx": {
        "model": "ComplEx",
        # ComplEx (Trouillon et al., 2016) utilise des embeddings dans
        # l'espace des nombres complexes. L'asymétrie est modélisée via
        # la partie imaginaire — parfait pour playsFor, coachedBy, etc.
        "model_kwargs": {
            "embedding_dim": EMBEDDING_DIM,
        },
        "loss":    "SoftplusLoss",
        "color":   "#2980B9",
    },
}

print("2. Configuration des modèles :")
for name, cfg in MODEL_CONFIGS.items():
    print(f"   {name:10s} | dim={EMBEDDING_DIM} | loss={cfg['loss']}")

In [ ]:
# =====================================================================
# --- 3. ENTRAÎNEMENT ---
# =====================================================================
results_store = {}

for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\n{'─'*55}")
    print(f"  Entraînement : {model_name}")
    print(f"{'─'*55}")
    t_start = time.time()

    result = pipeline(
        # Données
        training=tf_train,
        validation=tf_valid,
        testing=tf_test,
        # Modèle
        model=cfg["model"],
        model_kwargs=cfg["model_kwargs"],
        # Optimisation
        optimizer="Adam",
        optimizer_kwargs={"lr": LR},
        loss=cfg["loss"],
        # Entraînement
        training_kwargs={
            "num_epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
        },
        # Évaluation (calculée automatiquement sur test après training)
        evaluation_kwargs={"batch_size": BATCH_SIZE},
        # Reproductibilité
        random_seed=42,
        device=device,
    )

    elapsed = time.time() - t_start
    results_store[model_name] = {
        "pipeline_result": result,
        "elapsed":         elapsed,
    }

    # Résumé rapide
    metrics = result.metric_results.to_dict()
    mrr     = metrics.get("both.realistic.inverse_harmonic_mean_rank", 0)
    hits1   = metrics.get("both.realistic.hits_at_1", 0)
    hits10  = metrics.get("both.realistic.hits_at_10", 0)
    print(f"  ✅ {model_name} entraîné en {elapsed:.0f}s")
    print(f"     MRR      : {mrr:.4f}")
    print(f"     Hits@1   : {hits1:.4f}")
    print(f"     Hits@10  : {hits10:.4f}")

    # Sauvegarde du modèle
    save_dir = f"kge_{model_name.lower()}"
    result.save_to_directory(save_dir)
    print(f"     Sauvegardé : {save_dir}/")

In [ ]:
# =====================================================================
# --- 4. TABLEAU COMPARATIF DES MÉTRIQUES ---
# =====================================================================
print("\n4. Tableau comparatif des métriques (jeu de test)...\n")

METRIC_KEYS = {
    "MRR"     : "both.realistic.inverse_harmonic_mean_rank",
    "Hits@1"  : "both.realistic.hits_at_1",
    "Hits@3"  : "both.realistic.hits_at_3",
    "Hits@10" : "both.realistic.hits_at_10",
    "MR"      : "both.realistic.mean_rank",
}

rows = []
for model_name, data in results_store.items():
    metrics = data["pipeline_result"].metric_results.to_dict()
    row = {"Modèle": model_name}
    for label, key in METRIC_KEYS.items():
        val = metrics.get(key, float("nan"))
        row[label] = round(float(val), 4)
    row["Temps (s)"] = round(data["elapsed"], 1)
    rows.append(row)

df_metrics = pd.DataFrame(rows).set_index("Modèle")
print(df_metrics.to_string())

# Sauvegarde CSV
df_metrics.to_csv("kge_metrics_comparison.csv")
print("\n   -> kge_metrics_comparison.csv sauvegardé")

In [ ]:
# =====================================================================
# --- 5. VISUALISATION : COURBES DE LOSS ---
# =====================================================================
print("\n5. Visualisation des courbes de loss...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("C.3 — KGE Training Loss : TransE vs ComplEx", fontsize=14, fontweight="bold")

for ax, (model_name, data) in zip(axes, results_store.items()):
    losses = data["pipeline_result"].losses
    color  = MODEL_CONFIGS[model_name]["color"]
    ax.plot(losses, color=color, linewidth=1.5, label=model_name)
    ax.set_title(model_name, fontsize=12)
    ax.set_xlabel("Époque")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    # Annotation de la loss finale
    ax.annotate(
        f"Final : {losses[-1]:.4f}",
        xy=(len(losses)-1, losses[-1]),
        xytext=(-60, 20), textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="gray"),
        fontsize=9, color=color
    )

plt.tight_layout()
plt.savefig("kge_training_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("   -> kge_training_loss.png sauvegardé")

In [ ]:
# =====================================================================
# --- 6. RÉSUMÉ C.3 ---
# =====================================================================
print(f"\n{'='*60}")
print("✅  C.3 TERMINÉ — Résumé de l'entraînement KGE")
print(f"{'='*60}")
print(f"  Entités    : {tf_train.num_entities:,}")
print(f"  Relations  : {tf_train.num_relations:,}")
print(f"  Train      : {tf_train.num_triples:,} triplets")
print(f"  Dim. embed : {EMBEDDING_DIM}")
print(f"  Epochs     : {EPOCHS}")
print()
for model_name in results_store:
    metrics = results_store[model_name]["pipeline_result"].metric_results.to_dict()
    mrr   = metrics.get("both.realistic.inverse_harmonic_mean_rank", 0)
    h1    = metrics.get("both.realistic.hits_at_1", 0)
    h10   = metrics.get("both.realistic.hits_at_10", 0)
    t     = results_store[model_name]["elapsed"]
    print(f"  {model_name:10s} | MRR={mrr:.4f} | H@1={h1:.4f} | H@10={h10:.4f} | {t:.0f}s")

print(f"\n  Fichiers générés :")
print(f"   • kge_transe/       — modèle TransE sauvegardé")
print(f"   • kge_complex/      — modèle ComplEx sauvegardé")
print(f"   • kge_metrics_comparison.csv")
print(f"   • kge_training_loss.png")
print(f"\n  → Prochaine étape : C.4 — Évaluation approfondie & t-SNE")

Structure du C.3 :  
- Étape 1 — Chargement des splits avec TriplesFactory.from_path(). Le point clé est de partager entity_to_id et relation_to_id du train vers valid/test — c'est ce qui garantit l'absence d'entités OOV (déjà géré en C.1, mais PyKEEN l'enforces ici).
- Étape 2 — Configuration : deux modèles avec des approches fondamentalement différentes. TransE est translationnel (le plus simple, bon baseline), ComplEx est bilinéaire complexe (meilleur sur les relations asymétriques comme playsFor ou coachedBy).
- Étape 3 — pipeline() : c'est la fonction centrale de PyKEEN — elle orchestre entraînement + évaluation en une seule ligne. On passe directement les TriplesFactory, pas des chemins de fichiers.
- Étape 4 & 5 — Tableau comparatif et courbes de loss, qui seront utiles pour le rapport (section C.4 du grading).  


Quand tu as les résultats, on attaque C.4 — analyse approfondie des métriques + sensibilité à la taille + t-SNE.